In [ ]:
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits import mplot3d
from tqdm import tqdm
import random
from sklearn.preprocessing import StandardScaler
import math
import sys

# Without Labels

In [ ]:
data = h5py.File('../voxel_data/example_run_0017_clusters.h5', 'r')
DATAFRAME_PATH = '../voxel_data/example_run_0017_fit_2H.parquet'
labels_df = pd.read_parquet(DATAFRAME_PATH)[['polar', 'azimuthal', 'brho', 'event', 'cluster_index']]

root_group = data['cluster']
min_event = root_group.attrs.get("min_event")
max_event = root_group.attrs.get("max_event")
event_nums = range(0, max_event + 1)

# Initialize event_total_points array with zeros
event_total_points = np.zeros(max_event + 1, dtype=int)

# Iterate over events and fill in the total points
for event in event_nums:
    if f"event_{event}" in root_group.keys():
        total_points = 0
        for i in range(root_group[f"event_{event}"].attrs.get('nclusters')):
            cloud = root_group[f"event_{event}"][f"cluster_{i}"]["cloud"]
            num_points = cloud.shape[0]  # Number of points in the cluster
            track_labels = labels_df.loc[(labels_df["event"] == event) & (labels_df["cluster_index"] == i)]
            if not track_labels.empty:
                total_points += num_points
        event_total_points[event] = total_points

# Save the event_total_points array
np.save('../voxel_data/C16_event_lens.npy', event_total_points)

In [ ]:
file_name = 'C16_w_key_index'
max_points = np.max(event_total_points)
event_data = np.zeros((len(event_total_points), max_points, 5), float)

for n, event in tqdm(enumerate(range(len(event_total_points)))):
    event_key = f"event_{event}"
    if event_key in root_group.keys():
        event_group = root_group[event_key]
        cluster_count = event_group.attrs.get('nclusters')
        
        point_index = 0
        for i in range(cluster_count):
            cluster_key = f"cluster_{i}"
            if cluster_key in event_group.keys():
                cloud = event_group[cluster_key]["cloud"]
                num_points = cloud.shape[0]
                #print(cloud[:,0])
                #instant = np.zeros((num_points, 4))
                #instant[:, :cloud.shape[1]] = cloud[:, :4]

                track_labels = labels_df.loc[(labels_df["event"] == event) & (labels_df["cluster_index"] == i)]
                if not track_labels.empty:
                    track_labels = np.tile(np.array(track_labels[['polar', 'azimuthal', 'brho']]).flatten(), (num_points, 1))
                    if track_labels.shape[0] == num_points:
                        cloud = np.concatenate((cloud, track_labels), axis=1)
                        event_data[n, point_index:point_index + num_points, :4] = cloud[:, :4]
                        event_data[n, point_index:point_index + num_points, 4] = event
                        point_index += num_points

        event_data[n, 0, 4] = float(event)
                
                #event_data[n][point_index:point_index + num_points, :4] = cloud[:,:4]
                #event_data[n][point_index:point_index + num_points, 4] = float(event)
                
                #point_index += num_points

    event_data[n][0, 4] = float(event)

# Save the event data to a file
np.save('../voxel_data/' + file_name, event_data)
np.save('../voxel_data/' + 'C16_w_event_keys', event_data)

# With Labels

In [ ]:
data = h5py.File('../voxel_data/example_run_0017_clusters.h5', 'r')
DATAFRAME_PATH = '../voxel_data/example_run_0017_fit_2H.parquet'
labels_df = pd.read_parquet(DATAFRAME_PATH)[['polar', 'azimuthal', 'brho', 'event', 'cluster_index']]

root_group = data['cluster']
min_event = root_group.attrs.get("min_event")
max_event = root_group.attrs.get("max_event")
event_nums = range(0, max_event + 1)

# Initialize event_total_points array with zeros
event_total_points = np.zeros(max_event + 1, dtype=int)

# Iterate over events and fill in the total points
for event in event_nums:
    if f"event_{event}" in root_group.keys():
        total_points = 0
        for i in range(root_group[f"event_{event}"].attrs.get('nclusters')):
            cloud = root_group[f"event_{event}"][f"cluster_{i}"]["cloud"]
            num_points = cloud.shape[0]  # Number of points in the cluster
            track_labels = labels_df.loc[(labels_df["event"] == event) & (labels_df["cluster_index"] == i)]
            if not track_labels.empty:
                total_points += num_points
        event_total_points[event] = total_points

# Save the event_total_points array
np.save('../voxel_data/C16_event_lens.npy', event_total_points)

In [ ]:
file_name = 'C16_w_key_index'
max_points = np.max(event_total_points)
event_data = np.zeros((len(event_total_points), max_points, 5), float)

for n, event in tqdm(enumerate(range(len(event_total_points)))):
    event_key = f"event_{event}"
    if event_key in root_group.keys():
        event_group = root_group[event_key]
        cluster_count = event_group.attrs.get('nclusters')
        
        point_index = 0
        for i in range(cluster_count):
            cluster_key = f"cluster_{i}"
            if cluster_key in event_group.keys():
                cloud = event_group[cluster_key]["cloud"]
                num_points = cloud.shape[0]
                #print(cloud[:,0])
                #instant = np.zeros((num_points, 4))
                #instant[:, :cloud.shape[1]] = cloud[:, :4]

                track_labels = labels_df.loc[(labels_df["event"] == event) & (labels_df["cluster_index"] == i)]
                if not track_labels.empty:
                    track_labels = np.tile(np.array(track_labels[['polar', 'azimuthal', 'brho']]).flatten(), (num_points, 1))
                    if track_labels.shape[0] == num_points:
                        cloud = np.concatenate((cloud, track_labels), axis=1)
                        event_data[n, point_index:point_index + num_points, :4] = cloud[:, :4]
                        event_data[n, point_index:point_index + num_points, 4] = event
                        point_index += num_points

        event_data[n, 0, 4] = float(event)
                
                #event_data[n][point_index:point_index + num_points, :4] = cloud[:,:4]
                #event_data[n][point_index:point_index + num_points, 4] = float(event)
                
                #point_index += num_points

    event_data[n][0, 4] = float(event)

# Save the event data to a file
np.save('../voxel_data/' + file_name, event_data)
np.save('../voxel_data/' + 'C16_w_event_keys', event_data)